In [ ]:
import os, sys
sys.path.append('../src')
from FIT_python.data_split_and_summary.datasplit_and_summary_wraper import prepare_all_splits

# Daten-Splits sind bereits vorhanden, Aufruf bei Bedarf:
#prepare_all_splits()

In [ ]:
from FIT_python.plot_style import apply_style

apply_style()
import pandas as pd
from pathlib import Path
from FIT_python.pipeline_individual_id.utils import sequential_holdout_ids
from FIT_python.pipeline_individual_id.geometric_pairwise_projection import (
    generate_pairwise_comparisons_from_df,
)
from FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaseline
from FIT_python.pipeline_individual_id.supervised_umap_pipeline import run as run_umap

from FIT_python.pipeline_individual_id.siamese_pipeline import run as run_siamese
from FIT_python.pipeline_individual_id.evaluation import compute_confusion
import seaborn as sns, matplotlib.pyplot as plt
from FIT_python.data_split_and_summary.data_import_utils import (
    load_and_prep_df_individual,
    get_feature_cols,
)
from FIT_python.config import RESULTS_DATA_DIR, SPLITS_DIR

In [ ]:
splits_root = Path("data/splits/eurasian_otter")
models_root = (
    RESULTS_DATA_DIR
    / "eurasian_otter_random_search_standard_metrics"
    / "best_maj_test_pct"
)
output_root = Path("results/data/pairwise_comp_eurasian_otter")
output_root.mkdir(parents=True, exist_ok=True)

# 6) Loop über Spezies
species_dir = Path(SPLITS_DIR) / "eurasian_otter"
species = "eurasian_otter"
print(f"\n🔄 Processing species: {species}")

# 6.1) Train/Test laden & Fold entfernen
df = load_and_prep_df_individual(
    species, Path(SPLITS_DIR)
)  # falls hier bugs valid vales f und m checken

feature_cols = [
    c for c in df.columns if c.startswith(("dist", "ang", "t", "v")) and c != "trail"
]
ids = df["individual_id"].unique()
split = sequential_holdout_ids(ids, val_sizes=[4, 8, 12,16,20], n_iter=5, random_state=0)[0]
train_df = df[df["individual_id"].isin(split["train_ids"])]
val_df = df[df["individual_id"].isin(split["val_ids"])]
val_comps = generate_pairwise_comparisons_from_df(
    val_df,
    id_col="individual_id",
    group_sizes=[ 2,3,5, 7],
    n_repeats=4,
    mode="both",
    selfmatch_factor=2.0,
    n_folds=3,
    random_state=42,
    show_progress=True,
)

# In ein DataFrame überführen
df_val_comps = pd.DataFrame(val_comps)

In [ ]:
val_comps

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaseline
from FIT_python.pipeline_individual_id.evaluation import compute_confusion

splits = sequential_holdout_ids(ids, val_sizes=[3, 5, 9], n_iter=10, random_state=0)
cms_base, cms_umap, cms_siam = [], [], []

for split in splits:
    train_df = df[df["individual_id"].isin(split["train_ids"])]
    val_df = df[df["individual_id"].isin(split["val_ids"])]
    val_comps = generate_pairwise_comparisons_from_df(
        val_df, id_col="individual_id", group_sizes=[2,4,6],
        n_repeats=2, mode="both", selfmatch_factor=1.0,
        n_folds=2, random_state=42, show_progress=True,
    )
    res_base = DistanceBaseline.run(train_df, val_comps, feature_cols, val_df=val_df)
    Xb = res_base.filter(regex="^dist_")
    yb = res_base["same_individual"].map({"True": 1, "False": 0})
    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
    lr = LogisticRegression(max_iter=200)
    pred_b = cross_val_predict(lr, Xb, yb, cv=cv)
    cm_b = compute_confusion(pd.DataFrame({"same_individual": yb, "pred": pred_b}))
    cms_base.append(cm_b)

    res_umap, _ = run_umap(train_df, val_comps, feature_cols, val_df=val_df)
    res_umap = res_umap[res_umap["same_individual"].isin([True, False, 'True', 'False', 1, 0])]
    res_umap["pred"] = (res_umap["pred_same_proba"] > 0.5).astype(int)
    cm_u = compute_confusion(res_umap, true_col="same_individual", pred_col="pred")
    cms_umap.append(cm_u)

    res_siam = run_siamese(train_df, val_comps, feature_cols, val_df=val_df)
    df_siam = pd.DataFrame({
        'same_individual': [c['same_individual'] for c in val_comps],
        'pred_same': [r['pred_same'] for r in res_siam],
    })
    df_siam = df_siam[df_siam['same_individual'].isin([True, False, 'True', 'False', 1, 0])]
    df_siam['pred'] = (df_siam['pred_same'] > 0.5).astype(int)
    cm_s = compute_confusion(df_siam, true_col='same_individual', pred_col='pred')
    cms_siam.append(cm_s)

def sum_cms(cms):
    total = cms[0].copy()
    for cm in cms[1:]:
        total += cm
    return total

cm_base_total = sum_cms(cms_base)
cm_umap_total = sum_cms(cms_umap)
cm_siam_total = sum_cms(cms_siam)

display(cm_base_total, cm_umap_total, cm_siam_total)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
sns.heatmap(cm_base_total, annot=True, fmt='d', ax=axes[0])
axes[0].set_title('Baseline')
sns.heatmap(cm_umap_total, annot=True, fmt='d', ax=axes[1])
axes[1].set_title('Supervised UMAP')
sns.heatmap(cm_siam_total, annot=True, fmt='d', ax=axes[2])
axes[2].set_title('Siamese')
fig.tight_layout()
plt.show()

In [ ]:
# Load the test set and prepare trails without generating new windows
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

test_df = pd.read_parquet(species_dir / "test.parquet")
test_df = test_df.drop(columns=["Fold"], errors="ignore")

trails_per_animal = {}
for ind, grp_ind in test_df.groupby("individual_id"):
    trails_per_animal[ind] = {}
    for idx, (trail_id, g_trail) in enumerate(grp_ind.groupby("trail")):
        ids_list = g_trail["id"].astype(str).tolist()
        size = len(ids_list)
        trails_per_animal[ind].setdefault(size, []).append((idx, ids_list))

In [ ]:
# Generate pairwise comparisons using predefined trails

test_comps, _ = generate_pairwise_comparisons_from_df(
    test_df,
    id_col="individual_id",
    sampling_mode="predefined",
    trails_per_animal=trails_per_animal,
    n_folds=1,
    random_state=42,
)

df_test_comps = pd.DataFrame(test_comps)

In [ ]:
# Run inference on the test set
res_base_test = DistanceBaseline.run(train_df, test_comps, feature_cols, val_df=test_df)
res_umap_test, _ = run_umap(train_df, test_comps, feature_cols, val_df=test_df)
res_siam_test = run_siamese(train_df, test_comps, feature_cols, val_df=test_df)

In [ ]:
# Helper function to plot hierarchical clustering with Ward linkage

def plot_clustering(res_df, method_name):
    dists = res_df[["trail_a_id", "trail_b_id", "dist_euclidean"]]
    trails = sorted(set(dists["trail_a_id"]).union(dists["trail_b_id"]))
    index = {t: i for i, t in enumerate(trails)}
    mat = np.zeros((len(trails), len(trails)))
    for _, row in dists.iterrows():
        i = index[row.trail_a_id]
        j = index[row.trail_b_id]
        mat[i, j] = row.dist_euclidean
        mat[j, i] = row.dist_euclidean

    Z = linkage(squareform(mat), method="ward")

    trail_to_ind = {c["trail_a_id"]: c["ind_a"] for c in test_comps}
    trail_to_ind.update({c["trail_b_id"]: c["ind_b"] for c in test_comps})
    inds = sorted(set(trail_to_ind[t] for t in trails))
    palette = sns.color_palette("husl", len(inds))
    color_map = dict(zip(inds, palette))

    dendrogram(Z, labels=trails, orientation="top", above_threshold_color="grey")
    ax = plt.gca()
    for lbl in ax.get_xmajorticklabels():
        ind = trail_to_ind.get(lbl.get_text())
        if ind is not None:
            lbl.set_color(color_map[ind])
    ax.set_ylabel("distance")
    plt.title(f"{method_name} clustering")
    plt.show()

In [ ]:
# Plot clusters for all methods
plot_clustering(res_base_test, "Baseline")
plot_clustering(res_umap_test, "Supervised UMAP")
res_siam_df = pd.DataFrame({
    "trail_a_id": [c["trail_a_id"] for c in test_comps],
    "trail_b_id": [c["trail_b_id"] for c in test_comps],
    "dist_euclidean": [1 - r["pred_same"] for r in res_siam_test],
})
plot_clustering(res_siam_df, "Siamese")